# NB-05_salida_cg_coy_cgr

Process Flow SAS: **Salida CG coy_CGR** — `ProcessFlowContainer-4KYpcXdgqrX9yPA2`

In [ ]:
# ========= Parámetros =========
# Variables macro del SAS original. El .egp NO las define (venían del
# entorno SAS): su valor sale de la entrevista B4 o de
# project_config.yaml → run.macro_params, o se inyecta acá
# (celda 'parameters' de papermill).

Year = None  # &Year — nadie declaró su valor

faltantes = [n for n, v in {"Year": Year}.items() if v is None]
if faltantes:
    raise ValueError(f"Parámetros sin valor: {faltantes}")

In [ ]:
# ========= Celda 1: Configuración =========
import pandas as pd
import numpy as np
import os
import sqlalchemy
import datetime
from pathlib import Path
from sqlalchemy import text

# Conexión a BD — la define el orquestador al ejecutar
engine = sqlalchemy.create_engine(os.environ["SASMIG_DB_URL"], pool_pre_ping=True)
# Sesión de BD del notebook — espejo de la sesión WORK de SAS: las
# tablas temporales #tmp viven en ESTA conexión y mueren al cerrar el
# kernel. AUTOCOMMIT: cada statement commitea, como los pasos de SAS.
work_conn = engine.connect().execution_options(isolation_level="AUTOCOMMIT")

# Logging liviano de resultados — aprobado en la entrevista (Fase 4)
_LOG_PATH = Path("log") / "NB-05_salida_cg_coy_cgr.log"
_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
def _log(label, value=None):
    """Una línea por celda: imprime y persiste. Jamás rompe la corrida."""
    try:
        if hasattr(value, "shape"):
            detail = f"{value.shape[0]} filas x {value.shape[1]} cols"
        elif isinstance(value, int):
            detail = f"{value} filas"
        elif value is None:
            detail = "ok"
        else:
            detail = str(value)
        line = f"[{datetime.datetime.now():%Y-%m-%d %H:%M:%S}] {label}: {detail}"
        print(line)
        with open(_LOG_PATH, "a", encoding="utf-8") as fh:
            fh.write(line + "\n")
    except Exception:
        pass  # el log nunca puede tumbar el notebook
with open(_LOG_PATH, "a", encoding="utf-8") as _fh:
    _fh.write(f"\n=== corrida {datetime.datetime.now():%Y-%m-%d %H:%M:%S} ===\n")


## 1- TCons Mixtas_M_CGR

Distribuye la ejecución del año de las transacciones mixtas —la muestra y el resto expandido por ponderadores— hacia clasificación SCN, agente y uso, aplica el ajuste manual de D63 de 2020 y entrega la apertura desagregada por entidad, mes y trimestre

*confianza: medium · verificador: approve · SAS: PROC SQL CREATE TABLE (LEFT JOIN + GROUP BY) + DATA step SET/DROP + DELETE*

In [ ]:
# ========= 1- TCons Mixtas_M_CGR =========
# /*FILTRA DATOS DE EJECUCION CGR DEL AÑO*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cgr_y"))
sql_ejec_cgr_y = f"""
SELECT *
INTO #ejec_cgr_y
FROM GOBGENER.dbo.EJECUCION
WHERE [AÑO] = {int(Year)}
"""
res = work_conn.execute(text(sql_ejec_cgr_y))
n_ejec_cgr_y = res.rowcount
# DATA EJEC_CGR_Y; SET EJEC_CGR_Y (DROP=...) -> se quitan las mismas columnas de la temporal
work_conn.execute(text(
    "ALTER TABLE #ejec_cgr_y DROP COLUMN "
    "PPTO_INICIAL, PPTO_ACTUAL, PERCIB_PAG, POR_PERCIB_PAG, "
    "N_ITEM, N_ASIGNACION, C_SUBASIGNACION, N_SUBASIGNACION"
))
# t2.* de los PROC SQL: T-SQL no admite * junto a GROUP BY, se expande con las
# columnas que realmente quedaron en la temporal
cols_ejec = list(pd.read_sql(text("SELECT TOP 0 * FROM #ejec_cgr_y"), work_conn).columns)
sel_t2 = ", ".join(f"t2.[{c}]" for c in cols_ejec)
_log("#ejec_cgr_y", n_ejec_cgr_y)


In [ ]:
# /* CREA TABLA DE REDISTRIBUCION DE MUESTRA DE TRANSACCIONES MIXTAS (M) SEGUN ESTRUCTURA, PARA EJECUCION AÑO t */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mixtas_m"))
sql_ejec_muestra_mixtas_m = f"""
SELECT {sel_t2},
       t1.C_SCN,
       t1.C_CAGENTE,
       t1.C_CUP,
       SUM(t2.DEVENG / 1000000.0 * t1.POND) AS DEVENGO
INTO #ejec_muestra_mixtas_m
FROM GOBGENER.dbo.EST_MIXTAS_M t1
LEFT JOIN #ejec_cgr_y t2
       ON t2.C_ENTIDAD = t1.ENTIDAD
      AND t2.C_ASIGNACION = t1.C_ASIGNACION
      AND t2.C_ANALITICO = t1.C_ANALITICO
      AND t2.C_CUENTA = t1.C_CUENTA
      AND t2.C_ITEM = t1.C_ITEM
      AND t2.MES = t1.MES
WHERE t2.C_CUENTA IN ('24', '33')
  AND t2.C_ITEM IN ('03', '01', '06', '08', '09')
  AND t1.ENTIDAD NOT IN (5601, 10201)
  AND t2.DEVENG IS NOT NULL
GROUP BY {sel_t2}, t1.C_SCN, t1.C_CAGENTE, t1.C_CUP
"""
res = work_conn.execute(text(sql_ejec_muestra_mixtas_m))
n_muestra = res.rowcount
# DATA EJEC_MUESTRA_MIXTAS_M; SET EJEC_MUESTRA_MIXTAS_M (DROP=DEVENG)
work_conn.execute(text("ALTER TABLE #ejec_muestra_mixtas_m DROP COLUMN DEVENG"))
_log("#ejec_muestra_mixtas_m", n_muestra)


In [ ]:
# /* CREA TABLA DE ANALITICOS DE MIXTAS (M) A EXPANDIR (FUERA DE MUESTRA), SEGUN EJECUCION AÑO t */
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_ejec_exp_mixtas_m"))
# el SAS pide además t1.C_CUENTA/C_ITEM/C_ASIGNACION/C_ANALITICO: son duplicados de
# t2.* (equi-join) y SAS los descarta, así que solo se agrega Clasif_Tcons
sql_cruce_ejec_exp_mixtas_m = f"""
SELECT {sel_t2},
       t1.Clasif_Tcons
INTO #cruce_ejec_exp_mixtas_m
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN #ejec_cgr_y t2
       ON t2.C_ENTIDAD = t1.ENTIDAD
      AND t2.C_ASIGNACION = t1.C_ASIGNACION
      AND t2.C_ANALITICO = t1.C_ANALITICO
      AND t2.C_CUENTA = t1.C_CUENTA
      AND t2.C_ITEM = t1.C_ITEM
WHERE t1.M_MUESTRA = 0
  AND t2.C_ENTIDAD NOT IN (5601, 10201)
  AND t1.Clasif_Tcons = 'M'
  AND t2.DEVENG IS NOT NULL
"""
res = work_conn.execute(text(sql_cruce_ejec_exp_mixtas_m))
sel_t2_cruce = sel_t2 + ", t2.[Clasif_Tcons]"
_log("#cruce_ejec_exp_mixtas_m", res.rowcount)


In [ ]:
# /* CREA TABLA DE REDISTRIBUCION EXPANDIDA SEGUN ESTRUCTURA DE MUESTRA, PARA RESTO DE TRANSACCIONES MIXTAS (M),
# PARA EJECUCION AÑO t */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mixtas_m_prelim"))
sql_ejec_exp_mixtas_m_prelim = f"""
SELECT {sel_t2_cruce},
       t1.C_SCN,
       t1.C_CUP,
       t1.C_CAGENTE,
       SUM(t2.DEVENG) AS DATO_OR,
       SUM(CASE WHEN t3.DATO IS NOT NULL
                THEN t2.DEVENG / 1000000.0 * t1.POND * t3.DATO
                ELSE t2.DEVENG / 1000000.0 * t1.POND END) AS DEVENGO
INTO #ejec_exp_mixtas_m_prelim
FROM GOBGENER.dbo.EST_EXP_MIXTAS_M t1
LEFT JOIN #cruce_ejec_exp_mixtas_m t2
       ON t2.C_CUENTA = t1.C_CUENTA
      AND t2.C_ITEM = t1.C_ITEM
      AND t2.C_ASIGNACION = t1.C_ASIGNACION
      AND t2.C_ANALITICO = t1.C_ANALITICO
      AND t2.MES = t1.MES
      AND t2.C_ENTIDAD = t1.ENTIDAD
LEFT JOIN #prom_mes t3
       ON t2.MONEDA = t3.MONEDA
      AND t2.MES = t3.MES
      AND t2.[AÑO] = t3.[AÑO]
WHERE t2.DEVENG IS NOT NULL
GROUP BY {sel_t2_cruce}, t1.C_SCN, t1.C_CAGENTE, t1.C_CUP
"""
res = work_conn.execute(text(sql_ejec_exp_mixtas_m_prelim))
_log("#ejec_exp_mixtas_m_prelim", res.rowcount)


In [ ]:
# DATA EJEC_EXP_MIXTAS_M; SET EJEC_EXP_MIXTAS_M_prelim (DROP=DEVENG DATO_OR Clasif_Tcons)
cols_sin_deveng = [c for c in cols_ejec if c.upper() != "DEVENG"]
cols_mixtas = ", ".join(f"[{c}]" for c in cols_sin_deveng) + ", [C_SCN], [C_CAGENTE], [C_CUP], [DEVENGO]"
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mixtas_m"))
res = work_conn.execute(text(
    f"SELECT {cols_mixtas} INTO #ejec_exp_mixtas_m FROM #ejec_exp_mixtas_m_prelim"
))
_log("#ejec_exp_mixtas_m", res.rowcount)


In [ ]:
# /* Crea tabla con la ejecución total de transacciones Mixtas (M), Muestra + Resto */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mixtas_m_total"))
sql_ejec_mixtas_m_total = f"""
SELECT {cols_mixtas}
INTO #ejec_mixtas_m_total
FROM #ejec_muestra_mixtas_m
UNION ALL
SELECT {cols_mixtas}
FROM #ejec_exp_mixtas_m
"""
res = work_conn.execute(text(sql_ejec_mixtas_m_total))
_log("#ejec_mixtas_m_total", res.rowcount)


In [ ]:
# /* Ajuste por imputación de D63 en pandemia (2020) */
# el año, la entidad y el analítico van hardcodeados en el SAS: es un ajuste
# puntual y NO depende de Year
sql_ajuste_d63 = """
DELETE FROM #ejec_mixtas_m_total
WHERE C_ENTIDAD = 911
  AND C_CUENTA = '24'
  AND C_ITEM = '03'
  AND C_ASIGNACION = '002'
  AND C_ANALITICO IN ('03002000')
  AND [AÑO] = 2020
  AND MES > 4
"""
res = work_conn.execute(text(sql_ajuste_d63))
_log("DELETE #ejec_mixtas_m_total (ajuste D63 2020)", res.rowcount)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MIXTAS (M) para ANALISIS DESAGREGADO
#    CG coyuntura */
trim_case = (
    "CASE WHEN t1.MES IN (1,2,3) THEN 1 "
    "WHEN t1.MES IN (4,5,6) THEN 2 "
    "WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END"
)
act_case = (
    "CASE WHEN t3.ACTIVIDAD = 'EEPP' THEN 'EEPP' "
    "WHEN t3.ACTIVIDAD = 'SSPP' THEN 'SSPP' ELSE 'AAPP' END"
)
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_g_mixt_aper"))
sql_ejec_g_mixt_aper = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.[AÑO],
       {trim_case} AS [TRIM],
       t1.MES,
       t1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t1.C_SCN,
       t1.C_CAGENTE,
       SUM(t1.DEVENGO) AS Dato,
       t3.NIVEL,
       {act_case} AS C_ACT,
       'TC' AS C_ACTIVO
INTO #ejec_g_mixt_aper
FROM #ejec_mixtas_m_total t1
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t3 ON t1.C_ENTIDAD = t3.ENTIDAD
GROUP BY t1.SUBSECTOR,
         t1.MONEDA,
         t1.[AÑO],
         {trim_case},
         t1.MES,
         t1.C_ENTIDAD,
         t1.C_PARTIDA,
         t1.C_CAPITULO,
         t1.C_PROGRAMA,
         t1.C_TIPO_CUENTA,
         t1.C_CUENTA,
         t1.C_ITEM,
         t1.C_ASIGNACION,
         t1.C_ANALITICO,
         t1.N_ANALITICO,
         t1.C_SCN,
         t1.C_CAGENTE,
         t3.NIVEL,
         {act_case}
"""
res = work_conn.execute(text(sql_ejec_g_mixt_aper))
_log("#ejec_g_mixt_aper", res.rowcount)


In [ ]:
# PROC SQL DROP TABLE de las temporales de trabajo (el SAS las elimina al final);
# EJEC_CGR_Y y EJEC_G_MIXT_APER se conservan, igual que en el original
for _t in [
    "#ejec_muestra_mixtas_m",
    "#cruce_ejec_exp_mixtas_m",
    "#ejec_exp_mixtas_m_prelim",
    "#ejec_exp_mixtas_m",
    "#ejec_mixtas_m_total",
]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


## 4- Salida EJEC hasta CGR

Ejecución de gobierno central de coyuntura con datos de Contraloría, clasificada por transacción SCN y apertura institucional, con el ajuste de transferencias en especie y las imputaciones permanentes del año, en versión desagregada y agregada

*confianza: medium · verificador: approve · SAS: PROC SQL CREATE TABLE con LEFT JOIN múltiples + UPDATE/DELETE + PROC DATASETS APPEND + DATA step SET*

In [ ]:
# ========= 4- Salida EJEC hasta CGR =========
# Salida para indicadores GC de consumo de gobierno (coyuntura) DATA DE CONTRALORIA (t-1)
# Las expresiones que el SAS repite en cada subconjunto (A, B, C, D, F) se declaran
# una sola vez para que los seis bloques queden alineados.

# (CASE WHEN t4.ACTIVIDAD='FFAA' THEN 'AAPP' ... END) AS C_ACT
case_c_act = """(CASE WHEN t4.ACTIVIDAD = 'FFAA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'FFOO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'FONASA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'CORFO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'AAPP' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'SSPP' THEN 'SSPP'
             WHEN t4.ACTIVIDAD = 'EEPP' THEN 'EEPP' END)"""

# (CASE WHEN T3.DATO<>. THEN t1.DEVENG*t3.DATO/1000000 ELSE t1.DEVENG/1000000 END)
# FORMAT=COMMAX12.2 es formato de despliegue: no entra en el calculo
expr_dato = (
    "(CASE WHEN t3.DATO IS NOT NULL THEN t1.DEVENG * t3.DATO / 1000000.0"
    " ELSE t1.DEVENG / 1000000.0 END)"
)

# Transacciones SCN que arman el indicador de consumo de gobierno
scn_ind = "'D1','D11','D12','P2','P3','P72','D63','P1','P11'"

# Columnas de los subconjuntos que se unen al final (el UNION ALL exige alinearlas)
cols_ejec_id = (
    "SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, "
    "C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, "
    "C_SCN, C_CAGENTE, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL"
)
cols_ejec = cols_ejec_id + ", DATO"

# Columnas de EJECUCION_A_prelim (incluye ANALITICO_ y DEVENGO)
cols_prelim_id = (
    "SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, "
    "C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, ANALITICO_, "
    "C_SCN, C_CAGENTE, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL"
)
cols_prelim = cols_prelim_id + ", DEVENGO"

# Condicion del ajuste de sintesis: cuenta 22 / analitico 12999xxx / partida 16 / SSPP.
# COALESCE porque el faltante de SAS es blanco y en SQL Server llega como NULL
cond_aj = (
    "COALESCE(C_CUENTA, '') = '22' "
    "AND COALESCE(SUBSTRING(C_ANALITICO, 1, 5), '') = '12999' "
    "AND COALESCE(C_PARTIDA, '') = '16' "
    "AND COALESCE(C_ACT, '') = 'SSPP'"
)


In [ ]:
# /*Analiticos predefinidos */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim"))
sql_ejecucion_a_prelim = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1, 2, 3) THEN 1
             WHEN t1.MES IN (4, 5, 6) THEN 2
             WHEN t1.MES IN (7, 8, 9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       t1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       (CASE WHEN t2.ANALITICO = 'M' THEN t1.N_ANALITICO ELSE t2.ANALITICO END) AS ANALITICO_,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       {case_c_act} AS C_ACT,
       t4.NIVEL,
       {expr_dato} AS DEVENGO
INTO #ejecucion_a_prelim
FROM #ejec_cgr_y t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
    ON t1.C_TIPO_CUENTA = t2.T_CUENTA
   AND t1.C_CUENTA = t2.C_CUENTA
   AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
    ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE (t1.C_CUENTA IS NULL OR t1.C_CUENTA NOT IN ('05', '13', '24', '33'))
  AND t2.C_SCN IN ({scn_ind})
  AND (t1.C_ENTIDAD IS NULL OR t1.C_ENTIDAD NOT IN (5601, 10201))
"""
res = work_conn.execute(text(sql_ejecucion_a_prelim))
_log("#ejecucion_a_prelim", res.rowcount)


In [ ]:
# /*Este Ajuste lo pidieron de sintesis, incorpora al indicador de transferencias en esp
#   lo que consideran desde anuales, el porcentaje sale de anuales...*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_aj"))
sql_prelim_aj = f"""
SELECT {cols_prelim_id},
       (CASE WHEN {cond_aj} THEN DEVENGO * 0.928 ELSE DEVENGO END) AS DEVENGO
INTO #ejecucion_a_prelim_aj
FROM #ejecucion_a_prelim
"""
res = work_conn.execute(text(sql_prelim_aj))
_log("#ejecucion_a_prelim_aj", res.rowcount)


In [ ]:
# UPDATE EJECUCION_A_prelim_Aj: reclasifica las filas del ajuste a transferencia en especie
sql_update_aj = f"""
UPDATE #ejecucion_a_prelim_aj
SET C_SCN = 'D63',
    C_CAGENTE = 'S14',
    C_TRANSACCION = '433',
    T_TRANSACCION = 'transfesp',
    N_CUENTA = 'IG',
    C_CUP = '272'
WHERE {cond_aj}
"""
res = work_conn.execute(text(sql_update_aj))
_log("UPDATE #ejecucion_a_prelim_aj", res.rowcount)


In [ ]:
# CREATE TABLE EJECUCION_A_prelim AS SELECT ... DEVENGO*(1-0.928) ... FROM EJECUCION_A_prelim:
# el SAS rehace la tabla sobre si misma y solo cambia DEVENGO en las filas del ajuste
sql_prelim_resto = f"""
UPDATE #ejecucion_a_prelim
SET DEVENGO = DEVENGO * (1 - 0.928)
WHERE {cond_aj}
"""
res = work_conn.execute(text(sql_prelim_resto))
_log("UPDATE #ejecucion_a_prelim", res.rowcount)


In [ ]:
# CREATE TABLE EJECUCION_A_prelim_Aj AS SELECT ... WHERE <cond>: el SAS deja en _Aj
# solo las filas del ajuste (el NOT es seguro porque cond_aj usa COALESCE)
sql_prelim_aj_filtro = f"DELETE FROM #ejecucion_a_prelim_aj WHERE NOT ({cond_aj})"
res = work_conn.execute(text(sql_prelim_aj_filtro))
_log("DELETE #ejecucion_a_prelim_aj", res.rowcount)


In [ ]:
# PROC DATASETS; APPEND BASE=EJECUCION_A_prelim DATA=EJECUCION_A_prelim_Aj FORCE
sql_append_prelim = f"""
INSERT INTO #ejecucion_a_prelim ({cols_prelim})
SELECT {cols_prelim}
FROM #ejecucion_a_prelim_aj
"""
res = work_conn.execute(text(sql_append_prelim))
_log("APPEND #ejecucion_a_prelim", res.rowcount)


In [ ]:
# /*CALCULA RATIOS QUE SE USAN CON INFO DIPRES*/
work_conn.execute(text("DROP TABLE IF EXISTS #totales_inst"))
sql_totales_inst = """
SELECT t1.SUBSECTOR, t1.AÑO, t1.MONEDA, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
       t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_SCN, t1.C_CAGENTE,
       t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
       t1.C_ACT, t1.NIVEL,
       SUM(t1.DEVENGO) AS TOTAL_INST
INTO #totales_inst
FROM #ejecucion_a_prelim t1
WHERE t1.C_CUENTA = '22' AND t1.C_PARTIDA = '16' AND t1.C_ACT = 'SSPP'
GROUP BY t1.SUBSECTOR, t1.AÑO, t1.MONEDA, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
         t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_SCN, t1.C_CAGENTE,
         t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
         t1.C_ACT, t1.NIVEL
"""
res = work_conn.execute(text(sql_totales_inst))
_log("#totales_inst", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #totales"))
sql_totales = """
SELECT t1.SUBSECTOR, t1.AÑO, t1.MONEDA, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
       t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ACT, t1.NIVEL,
       SUM(t1.TOTAL_INST) AS TOTAL
INTO #totales
FROM #totales_inst t1
GROUP BY t1.SUBSECTOR, t1.AÑO, t1.MONEDA, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
         t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ACT, t1.NIVEL
"""
res = work_conn.execute(text(sql_totales))
_log("#totales", res.rowcount)


In [ ]:
# /****LLEVAR TABLA RATIOS A SALIDA EN EXCEL****/
# /*LLEVAR ESTA TABLA DE RATIOS A SALIDA EXCEL A CARPETA DE OUTPUTS INTERMEDIOS*/
work_conn.execute(text("DROP TABLE IF EXISTS #ratios"))
sql_ratios = """
SELECT t1.SUBSECTOR, t1.AÑO, t1.MONEDA, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
       t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_SCN, t1.C_CAGENTE,
       t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
       t1.C_ACT, t1.NIVEL,
       t1.TOTAL_INST / NULLIF(t2.TOTAL, 0) AS Ratio
INTO #ratios
FROM #totales_inst t1
LEFT JOIN #totales t2 ON t1.C_ENTIDAD = t2.C_ENTIDAD AND t1.AÑO = t2.AÑO
"""
res = work_conn.execute(text(sql_ratios))
_log("#ratios", res.rowcount)
# /*Fin de ajuste....*/


In [ ]:
sql_del_analiticos_32 = """
DELETE FROM #ejecucion_a_prelim
WHERE C_CUENTA = '32' AND C_ANALITICO IN ('04001000', '04002000')
"""
res = work_conn.execute(text(sql_del_analiticos_32))
_log("DELETE #ejecucion_a_prelim (cuenta 32)", res.rowcount)


In [ ]:
# /*Analiticos predefinidos: AJUSTE COYUNTURA - se elimina de indicador transaccion CORFO
#   que se desagrega anualmente entre ventas y D45 */
sql_del_corfo = """
DELETE FROM #ejecucion_a_prelim
WHERE C_ENTIDAD = 3801 AND C_CUENTA = '06' AND C_ITEM = '01'
  AND C_ASIGNACION = '000' AND C_ANALITICO IN ('01000000')
"""
res = work_conn.execute(text(sql_del_corfo))
_log("DELETE #ejecucion_a_prelim (CORFO)", res.rowcount)


In [ ]:
# /*Analiticos predefinidos (sigue)*/
# DATA EJECUCION_A: DATO=DEVENGO, se descartan DEVENGO y ANALITICO_ y se filtra por C_SCN
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a"))
sql_ejecucion_a = f"""
SELECT {cols_ejec_id},
       DEVENGO AS DATO
INTO #ejecucion_a
FROM #ejecucion_a_prelim
WHERE C_SCN IN ({scn_ind})
"""
res = work_conn.execute(text(sql_ejecucion_a))
_log("#ejecucion_a", res.rowcount)


In [ ]:
# /*Analiticos FIJOS CUENTA 32*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_b"))
sql_ejecucion_b = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1, 2, 3) THEN 1
             WHEN t1.MES IN (4, 5, 6) THEN 2
             WHEN t1.MES IN (7, 8, 9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
       t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP,
       {case_c_act} AS C_ACT,
       t4.NIVEL,
       {expr_dato} AS DATO
INTO #ejecucion_b
FROM #ejec_cgr_y t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
    ON t1.C_TIPO_CUENTA = t2.T_CUENTA
   AND t1.C_CUENTA = t2.C_CUENTA
   AND t1.C_ITEM = t2.C_ITEM
   AND t1.C_ASIGNACION = t2.C_ASIGNACION
   AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
    ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t1.C_CUENTA = '32'
  AND t2.C_SCN IN ({scn_ind})
  AND t1.C_ANALITICO IN ('04001000', '04002000')
  AND (t1.C_ENTIDAD IS NULL OR t1.C_ENTIDAD NOT IN (5601, 10201))
"""
res = work_conn.execute(text(sql_ejecucion_b))
_log("#ejecucion_b", res.rowcount)


In [ ]:
# /*analiticos variables*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_c"))
sql_ejecucion_c = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1, 2, 3) THEN 1
             WHEN t1.MES IN (4, 5, 6) THEN 2
             WHEN t1.MES IN (7, 8, 9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
       t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP,
       {case_c_act} AS C_ACT,
       t4.NIVEL,
       {expr_dato} AS DATO
INTO #ejecucion_c
FROM #ejec_cgr_y t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA
   AND t1.C_CAPITULO = t2.C_CAPITULO
   AND t1.C_PROGRAMA = t2.C_PROGRAMA
   AND t1.C_ENTIDAD = t2.ENTIDAD
   AND t1.C_TIPO_CUENTA = t2.T_CUENTA
   AND t1.C_CUENTA = t2.C_CUENTA
   AND t1.C_ITEM = t2.C_ITEM
   AND t1.C_ASIGNACION = t2.C_ASIGNACION
   AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
    ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t1.C_CUENTA IN ('05', '13', '24', '33')
  AND t2.C_SCN IN ({scn_ind})
  AND (t2.OBS IS NULL OR t2.OBS NOT IN ('CR18_difcoy', 'CR18_difact'))
  AND (t1.C_ENTIDAD IS NULL OR t1.C_ENTIDAD NOT IN (5601, 10201))
  AND (t2.C_SCN IS NULL OR t2.C_SCN <> 'TC')
"""
res = work_conn.execute(text(sql_ejecucion_c))
_log("#ejecucion_c", res.rowcount)


In [ ]:
# /* Transacciones diferenciadas para coyuntura */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_d"))
sql_ejecucion_d = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1, 2, 3) THEN 1
             WHEN t1.MES IN (4, 5, 6) THEN 2
             WHEN t1.MES IN (7, 8, 9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
       t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_ACTIVO AS C_SCN,
       t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP,
       {case_c_act} AS C_ACT,
       t4.NIVEL,
       {expr_dato} AS DATO
INTO #ejecucion_d
FROM #ejec_cgr_y t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA
   AND t1.C_CAPITULO = t2.C_CAPITULO
   AND t1.C_PROGRAMA = t2.C_PROGRAMA
   AND t1.C_ENTIDAD = t2.ENTIDAD
   AND t1.C_TIPO_CUENTA = t2.T_CUENTA
   AND t1.C_CUENTA = t2.C_CUENTA
   AND t1.C_ITEM = t2.C_ITEM
   AND t1.C_ASIGNACION = t2.C_ASIGNACION
   AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
    ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t2.OBS IN ('CR18_difcoy')
  AND (t1.C_ENTIDAD IS NULL OR t1.C_ENTIDAD NOT IN (5601, 10201))
  AND t2.C_ACTIVO IN ({scn_ind})
  AND (t2.C_SCN IS NULL OR t2.C_SCN <> 'TC')
"""
res = work_conn.execute(text(sql_ejecucion_d))
_log("#ejecucion_d", res.rowcount)


In [ ]:
# /* Transacciones diferenciadas por actividad */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_f"))
sql_ejecucion_f = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1, 2, 3) THEN 1
             WHEN t1.MES IN (4, 5, 6) THEN 2
             WHEN t1.MES IN (7, 8, 9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
       t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP,
       t2.C_CUP AS C_ACT,
       t4.NIVEL,
       {expr_dato} AS DATO
INTO #ejecucion_f
FROM #ejec_cgr_y t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA
   AND t1.C_CAPITULO = t2.C_CAPITULO
   AND t1.C_PROGRAMA = t2.C_PROGRAMA
   AND t1.C_ENTIDAD = t2.ENTIDAD
   AND t1.C_TIPO_CUENTA = t2.T_CUENTA
   AND t1.C_CUENTA = t2.C_CUENTA
   AND t1.C_ITEM = t2.C_ITEM
   AND t1.C_ASIGNACION = t2.C_ASIGNACION
   AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
    ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t2.OBS IN ('CR18_difact')
  AND (t1.C_ENTIDAD IS NULL OR t1.C_ENTIDAD NOT IN (5601, 10201))
  AND (t2.C_SCN IS NULL OR t2.C_SCN <> 'TC')
  AND t2.C_SCN IN ({scn_ind})
"""
res = work_conn.execute(text(sql_ejecucion_f))
_log("#ejecucion_f", res.rowcount)


In [ ]:
# /* Transacciones TCONS (solo aplica Mixtas) */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_g"))
sql_ejecucion_g = f"""
SELECT *
INTO #ejecucion_g
FROM #ejec_g_mixt_aper
WHERE C_SCN IN ({scn_ind})
"""
res = work_conn.execute(text(sql_ejecucion_g))
_log("#ejecucion_g", res.rowcount)


In [ ]:
# /*UNE TABLAS de Salida total de ejecucion CG coy de data CGR*/
# El DATA step concatena A, B, C, D, G, F y descarta ACTIVIDAD; el UNION ALL
# exige la lista de columnas explicita y alineada
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_cgr_prev"))
sql_cg_cgr_prev = f"""
SELECT {cols_ejec}
INTO #ejecucion_cg_cgr_prev
FROM (
    SELECT {cols_ejec} FROM #ejecucion_a
    UNION ALL
    SELECT {cols_ejec} FROM #ejecucion_b
    UNION ALL
    SELECT {cols_ejec} FROM #ejecucion_c
    UNION ALL
    SELECT {cols_ejec} FROM #ejecucion_d
    UNION ALL
    SELECT {cols_ejec} FROM #ejecucion_g
    UNION ALL
    SELECT {cols_ejec} FROM #ejecucion_f
) u
"""
res = work_conn.execute(text(sql_cg_cgr_prev))
_log("#ejecucion_cg_cgr_prev", res.rowcount)


In [ ]:
# DROP TABLE EJECUCION_A_prelim, EJECUCION_A, EJECUCION_B, EJECUCION_C,
# EJECUCION_D, EJECUCION_G, EJECUCION_F (EJECUCION_CG_DIP_imp no se crea en este nodo)
for _t in [
    "#ejecucion_a_prelim",
    "#ejecucion_a",
    "#ejecucion_b",
    "#ejecucion_c",
    "#ejecucion_d",
    "#ejecucion_g",
    "#ejecucion_f",
]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


In [ ]:
# /*Incorporacion de imputaciones permanentes*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_imput_perm"))
sql_imput_perm = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       t1.TRIM,
       t1.MES,
       t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
       t1.C_ANALITICO, t1.N_ANALITICO,
       t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
       t1.C_ACTIVO, t1.C_CUP,
       (CASE WHEN t1.C_ACT = 'FFAA' THEN 'AAPP'
             WHEN t1.C_ACT = 'FFOO' THEN 'AAPP'
             WHEN t1.C_ACT = 'PREVISION' THEN 'AAPP'
             WHEN t1.C_ACT = 'FONASA' THEN 'AAPP'
             WHEN t1.C_ACT = 'CORFO' THEN 'AAPP'
             WHEN t1.C_ACT = 'CENABAST' THEN 'AAPP'
             WHEN t1.C_ACT = 'AAPP' THEN 'AAPP'
             WHEN t1.C_ACT = 'SSPP' THEN 'SSPP'
             WHEN t1.C_ACT = 'EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       t1.DATO AS DATO
INTO #ejecucion_cg_dip_imput_perm
FROM GOBGENER.dbo.IMPUT_GC_DIP_APER t1
WHERE t1.AÑO = {int(Year)}
  AND (t1.C_SCN IS NULL OR t1.C_SCN NOT IN
       ('F22', 'F29', 'F3', 'F32', 'F41', 'F42', 'F5', 'F612', 'F7', 'F71', 'F79',
        'K10', 'K11', 'K9'))
"""
res = work_conn.execute(text(sql_imput_perm))
_log("#ejecucion_cg_dip_imput_perm", res.rowcount)


In [ ]:
# /*UNE TABLAS SALIDA DESAGREGADA*/
# /****LLEVAR TABLA EJECUCION_COY_CGR_APER A SALIDA EN EXCEL****/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_coy_cgr_aper"))
sql_coy_cgr_aper = f"""
SELECT {cols_ejec}
INTO #ejecucion_coy_cgr_aper
FROM (
    SELECT {cols_ejec} FROM #ejecucion_cg_cgr_prev
    UNION ALL
    SELECT {cols_ejec} FROM #ejecucion_cg_dip_imput_perm
) u
"""
res = work_conn.execute(text(sql_coy_cgr_aper))
_log("#ejecucion_coy_cgr_aper", res.rowcount)


In [ ]:
# /*TABLA SALIDA AGREGADA*/
# /****LLEVAR TABLA EJECUCION_COY_CGR A SALIDA EN EXCEL****/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_coy_cgr"))
sql_coy_cgr = """
SELECT t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
       t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL,
       SUM(t1.DATO) AS DATO
INTO #ejecucion_coy_cgr
FROM #ejecucion_coy_cgr_aper t1
GROUP BY t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
         t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
         t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL
"""
res = work_conn.execute(text(sql_coy_cgr))
_log("#ejecucion_coy_cgr", res.rowcount)


In [ ]:
# DROP TABLE EJECUCION_CG_DIP_IMPUT_PERM, totales_inst, totales
# (aux_ratios esta en el DROP del SAS pero no se crea en este nodo)
for _t in ["#ejecucion_cg_dip_imput_perm", "#totales_inst", "#totales"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))
